In [275]:
import pandas as pd
import numpy as np
# 행(row) 다 보기
pd.set_option('display.max_rows', None)

# 열(column) 다 보기
pd.set_option('display.max_columns', None)

In [281]:
dict_us_stocks = pd.read_pickle('yf_chunk_all.pkl')

df = pd.read_pickle('bs_fin.pickle')
df['per'] = df['per'].replace([np.inf, -np.inf], 0)
df['payout(%)'] = np.round(df['payout(%)'] * 100, 2)
df.head()

,Symbol,sector,industry,market_cap,price,eps,pbs,per,pbr,psr,roe,net_income,ev/ebitda,freecashflow,is_share_reduced,is_share_same,is_eps_inc,is_revenue_inc,is_operating_income_inc,is_net_income_inc,is_debt_inc,is_debt_long_inc,is_debt_short_inc,is_debt_payables_inc,is_debt_deferredTax,total_revenue,current(%),quick(%),op(%),net(%),dividend_rate,dividend_yield,payout(%),debt(%),debt_long(%),debt_short(%),debt_payables(%),debt_tax(%),is_ok_payables,is_ok_tax
0,CRESY,Industrials,Conglomerates,825891008,11.83,1.81,1.207939,6.535912,9.793539,0.000858,0.19706,3.684400e+10,13.853,-73069625344,False,False,True,True,True,False,True,True,True,True,False,3.185290e+11,133.07,101.43,19.12,11.57,0.63,5.32,23.40,305.27,34.19,15.09,12.52,0.0,True,True
1,SCHL,Communication Services,Publishing,728269824,28.97,-0.69,34.926000,0.000000,0.829468,0.451248,-0.01144,-7.110000e+07,13.513,69587504,False,False,False,False,False,False,True,True,True,False,False,2.256000e+08,115.65,64.62,-40.51,-31.52,0.80,2.76,135.59,122.62,38.11,0.58,16.33,0.0,True,True
2,GYRE,Healthcare,Biotechnology,740801984,7.69,0.04,1.121000,192.250000,6.859946,6.906279,0.10017,3.610000e+06,47.048,2900250,False,False,True,True,True,True,False,False,False,True,False,3.056400e+07,619.47,563.83,22.66,11.81,0.00,0.00,0.00,21.56,2.45,0.00,2.53,0.0,True,True
3,LINC,Consumer Defensive,Education & Training Services,696356032,22.02,0.45,5.877000,48.933334,3.746810,1.407513,0.07943,3.799000e+06,21.105,-36984376,False,False,False,True,True,False,True,True,False,True,False,1.413890e+08,80.66,77.43,4.45,2.69,0.00,0.00,0.00,151.24,65.89,0.00,12.48,0.0,True,True
4,EMBC,Healthcare,Medical Instruments & Supplies,690451520,11.80,1.62,-11.122000,7.283951,-1.060960,0.639070,0.00000,2.640000e+07,5.372,183150000,False,False,True,False,True,True,True,False,False,False,False,2.640000e+08,241.45,173.15,24.36,10.00,0.60,5.08,37.04,-267.52,81.36,0.55,4.26,0.0,True,True


### 1. 1차 선별 기준
- 작년 동기 대비 eps, 매출, 순이익 증가 했는가?
- 작년 동기 대비 빚은 줄였고 갚을 능력이 되는가?
- 순이익 비중이 20% 이상인가?

In [282]:
df_refined = df[(df['is_eps_inc'] == True)
                & (df['is_revenue_inc'] == True)
                & (df['is_net_income_inc'] == True)
                & (df['is_share_reduced'] == True)
                & (df['is_debt_inc'] == False)
                & (df['is_ok_payables'] == True)
                & (df['is_ok_tax'] == True)
                & (df['net(%)'] > 20)].sort_values(by=['market_cap', 'op(%)', 'net(%)'], ascending=False)

list_refined_tickers = df_refined['Symbol'].tolist()
df_candid = df_refined[['Symbol', 'sector', 'industry', 'price', 'market_cap', 'net_income', 'per']].copy()
df_refined

,Symbol,sector,industry,market_cap,price,eps,pbs,per,pbr,psr,roe,net_income,ev/ebitda,freecashflow,is_share_reduced,is_share_same,is_eps_inc,is_revenue_inc,is_operating_income_inc,is_net_income_inc,is_debt_inc,is_debt_long_inc,is_debt_short_inc,is_debt_payables_inc,is_debt_deferredTax,total_revenue,current(%),quick(%),op(%),net(%),dividend_rate,dividend_yield,payout(%),debt(%),debt_long(%),debt_short(%),debt_payables(%),debt_tax(%),is_ok_payables,is_ok_tax
196,NVDA,Technology,Semiconductors,4261212061696,175.02,4.04,4.892000,43.321785,35.776780,22.769940,1.07359,3.191000e+10,37.297,53282873344,True,False,True,True,True,True,False,False,True,True,False,5.700600e+10,446.76,370.88,63.17,55.98,0.04,0.02,0.99,35.54,22.44,2.36,20.41,0.0,True,True
197,AAPL,Technology,Consumer Electronics,4129783545856,278.28,7.46,4.991000,37.302948,55.756360,9.923524,1.71422,2.746600e+10,28.806,78862254080,True,False,True,True,True,True,False,False,False,True,False,1.024660e+11,89.33,85.88,31.65,26.80,1.04,0.37,13.67,387.22,27.43,9.91,24.47,0.0,True,True
198,MSFT,Technology,Software - Infrastructure,3556993138688,478.53,14.07,48.840000,34.010662,9.797912,12.106358,0.32241,2.774700e+10,21.479,53327376384,True,False,True,True,True,True,False,False,True,True,False,7.767300e+10,140.05,139.22,48.87,35.72,3.64,0.76,23.61,75.27,19.29,2.87,11.92,0.0,True,True
649,TSM,Technology,Semiconductors,1514666852352,292.04,9.75,6.175678,29.952822,47.288734,0.417100,0.34657,4.523014e+11,2.370,628505444352,True,False,True,True,True,True,False,True,False,True,False,9.899183e+11,269.30,246.67,50.58,45.69,3.37,1.15,29.79,46.39,40.94,0.00,11.21,0.0,True,True
650,LLY,Healthcare,Drug Manufacturers - General,921118310400,1027.51,20.45,26.560000,50.244987,38.686370,15.501875,0.96470,5.582500e+09,33.163,1397924992,True,False,True,True,True,True,False,True,False,True,False,1.760080e+10,154.63,124.29,48.29,31.72,6.23,0.61,28.42,382.82,44.87,1.79,4.68,0.0,True,True
201,ASML,Technology,Semiconductor Equipment & Materials,419529424896,1080.85,28.32,57.568250,38.165607,18.775106,13.024010,0.53852,2.124500e+09,34.232,9320262656,True,False,True,True,True,True,False,False,False,False,False,7.516000e+09,130.80,69.62,32.84,28.27,7.37,0.68,26.72,137.43,10.36,0.00,0.00,0.0,True,True
656,PG,Consumer Defensive,Household & Personal Products,334314930176,142.84,6.86,22.467000,20.822157,6.357769,3.936219,0.31902,4.750000e+09,14.546,12200999936,True,False,True,True,True,True,False,False,True,True,False,2.238600e+10,71.37,50.72,26.16,21.22,4.23,2.96,60.24,139.01,32.84,15.71,21.08,0.0,True,True
215,KLAC,Technology,Semiconductor Equipment & Materials,157220798464,1193.92,31.88,37.905000,37.450443,31.497694,12.553244,0.99168,1.121040e+09,28.129,2757365760,True,False,True,True,True,True,False,True,False,True,False,3.209696e+09,269.17,187.75,41.66,34.93,7.60,0.64,22.62,227.34,53.31,0.00,3.79,0.0,True,True
216,GILD,Healthcare,Drug Manufacturers - General,149393162240,120.40,6.47,17.343000,18.608965,6.942282,5.136080,0.40706,3.052000e+09,11.725,8028000256,True,False,True,True,True,True,False,True,True,False,False,7.769000e+09,145.35,130.84,45.01,39.28,3.16,2.62,48.61,172.13,59.70,7.57,2.18,0.0,True,True
219,SNY,Healthcare,Drug Manufacturers - General,118824140800,48.68,3.08,70.603010,15.805195,0.689489,2.587014,0.08835,2.802000e+09,5.871,0,True,False,True,True,True,True,False,False,False,True,False,1.317000e+10,106.03,29.72,28.74,21.28,2.20,4.51,73.86,76.79,23.58,14.58,39.10,0.0,True,True


### 2. 2차 선별 기준
- 동종 업계 대비 per가 낮아?
- 내 요구수익률보다 roe가 높아?
- 현금흐름창출이 좋아?
- 내가 이 회사를 산다 했을 때 영업을 통해 갚을 수 있는 년수는?
- roe 분해
    - 마진이 높으면 프리미엄/대세/성장주
    - 재고회전율이 높으면 싸게 많이 파는 효율형 회사
    - 레버리지가 크면 부채형(은신행/통신)

In [283]:
list_net_margin = []
list_inventory_turnover = []
list_asset_turnover = []
list_leverage = []
list_total_equity = []
list_operating_cash_flow = []
list_capital_expenditure = []
list_debt = []
list_cash = []
list_ebitda = []
for tkr in list_refined_tickers:
    ####################################################################
    # 얼마나 비싸게 파는가? (브랜드, 기술력, 독점력, 가격 결정력) 적게 팔아도 남는다
    ####################################################################
    net_income = dict_us_stocks[tkr]['financials']['Net Income'].iloc[0]
    total_revenue = dict_us_stocks[tkr]['financials']['Total Revenue'].iloc[0]
    net_margin = np.round(net_income / total_revenue, 2)

    ## 재고회전율
    ####################################################################
    # 얼마나 빨리 팔아치우는가? (유통, 물,공급망) 회전율이 높으면 돈이 묶이질 않음
    ####################################################################
    inventory = dict_us_stocks[tkr]['balance_sheet']['Inventory'].iloc[0]
    cost_of_revenue = dict_us_stocks[tkr]['incomestmt']['Cost Of Revenue'].iloc[0]
    inventory_turnover = np.round(cost_of_revenue / inventory, 2)

    ## 자산회전율
    ####################################################################
    # 이 회사는 가진 자산을 1년에 몇번이나 굴려서 매출을 만드냐?
    ####################################################################
    total_revenue = dict_us_stocks[tkr]['incomestmt']['Total Revenue'].iloc[0]
    total_assets = dict_us_stocks[tkr]['balance_sheet']['Total Assets'].iloc[0]
    asset_turnover = np.round(total_revenue / total_assets, 2)

    ## 레버리지
    ####################################################################
    # 내 돈 말고 얼마나 남의 돈을 쓰는가? (부채, 차입) 레버리지 높으면 ROE가 높아짐
    ####################################################################
    total_equity = dict_us_stocks[tkr]['balance_sheet']['Total Equity Gross Minority Interest'].iloc[0]
    leverage = np.round(total_assets / total_equity, 2)

    ####################################################################
    # 회사를 굴리고도 남은 진짜 현금  ( + 면 현금창출력, - 면 투자국면 이거나 비지니스 문제)
    ####################################################################
    operating_cash_flow = dict_us_stocks[tkr]['cashflow']['Operating Cash Flow'].iloc[0]
    capital_expenditure = dict_us_stocks[tkr]['cashflow']['Capital Expenditure'].iloc[0]  \
                            if 'Capital Expenditure' in dict_us_stocks[tkr]['cashflow'].columns else 0

    ####################################################################
    # ev/ebitda 회사를 샀을 때 회수하려면 몇년 걸리냐?
    ####################################################################
    debt = dict_us_stocks[tkr]['balance_sheet']['Total Debt'].iloc[0]
    cash = dict_us_stocks[tkr]['balance_sheet']['Cash And Cash Equivalents'].iloc[0]
    ebitda = dict_us_stocks[tkr]['incomestmt']['EBITDA'].iloc[0]

    list_net_margin.append(net_margin)
    list_inventory_turnover.append(inventory_turnover)
    list_asset_turnover.append(asset_turnover)
    list_leverage.append(leverage)
    list_total_equity.append(total_equity)
    list_operating_cash_flow.append(operating_cash_flow)
    list_capital_expenditure.append(capital_expenditure)
    list_debt.append(debt)
    list_cash.append(cash)
    list_ebitda.append(ebitda)

df_candid['net_margin'] = list_net_margin
df_candid['inventory_turnover'] = list_inventory_turnover
df_candid['asset_turnover'] = list_asset_turnover
df_candid['leverage'] = list_leverage
df_candid['total_equity'] = list_total_equity
df_candid['operating_cash_flow'] = list_operating_cash_flow
df_candid['capital_expenditure'] = list_capital_expenditure
df_candid['debt'] = list_debt
df_candid['cash'] = list_cash
df_candid['ebitda'] = list_ebitda


In [284]:
df_candid['roe'] = df_candid['net_margin'] * df_candid['asset_turnover'] * df_candid['leverage']
df_candid['pbr'] = np.round(df_candid['market_cap'] / df_candid['total_equity'], 2)
df_candid['fcf'] = df_candid['operating_cash_flow'] + df_candid['capital_expenditure']
df_candid['ev'] = df_candid['market_cap'] + df_candid['debt'] - df_candid['cash']

df_sector = (
    df.query("per > 0")
      .groupby(['sector', 'industry'])
      .per.mean()
      .reset_index(name='per_mean')
)

df_candid = df_candid.merge(df_sector, on=['sector', 'industry'], how='left')
df_candid['relative_per'] = np.round(df_candid['per'] / df_candid['per_mean'], 2)

df_candid['ev_ebitda'] = np.round(df_candid['ev'] / df_candid['ebitda'], 2)
df_candid['net_marketcap'] = np.round(df_candid['net_income'] / df_candid['market_cap'], 2) * 100
df_candid['fcf_marketcap'] = np.round(df_candid['fcf'] / df_candid['market_cap'], 2) * 100

df_candid.head()

,Symbol,sector,industry,price,market_cap,net_income,per,net_margin,inventory_turnover,asset_turnover,leverage,total_equity,operating_cash_flow,capital_expenditure,debt,cash,ebitda,roe,pbr,fcf,ev,per_mean,relative_per,ev_ebitda,net_marketcap,fcf_marketcap
0,NVDA,Technology,Semiconductors,175.02,4261212061696,3.191000e+10,43.321785,0.56,0.77,0.35,1.36,1.188970e+11,2.375100e+10,-1.636000e+09,1.048100e+10,1.148600e+10,3.874800e+10,0.266560,35.84,2.211500e+10,4.260207e+12,67.589582,0.64,109.95,1.0,1.0
1,AAPL,Technology,Consumer Electronics,278.28,4129783545856,2.746600e+10,37.302948,0.27,9.47,0.29,4.87,7.373300e+10,2.972800e+10,-3.242000e+09,9.865700e+10,3.593400e+10,3.555400e+10,0.381321,56.01,2.648600e+10,4.192507e+12,29.108505,1.28,117.92,1.0,1.0
2,MSFT,Technology,Software - Infrastructure,478.53,3556993138688,2.774700e+10,34.010662,0.36,21.28,0.12,1.75,3.630760e+11,4.505700e+10,-1.939400e+10,6.055600e+10,2.884900e+10,4.806000e+10,0.075600,9.80,2.566300e+10,3.588700e+12,18.821010,1.81,74.67,1.0,1.0
3,TSM,Technology,Semiconductors,292.04,1514666852352,4.523014e+11,29.952822,0.46,1.39,0.13,1.46,5.035578e+12,4.268291e+11,-2.884433e+11,9.492073e+11,2.470759e+12,6.911149e+11,0.087308,0.30,1.383857e+11,-6.885280e+09,67.589582,0.44,-0.01,30.0,9.0
4,LLY,Healthcare,Drug Manufacturers - General,1027.51,921118310400,5.582500e+09,50.244987,0.32,0.25,0.15,4.82,2.385080e+10,8.835900e+09,-2.808200e+09,4.250660e+10,9.791900e+09,7.882000e+09,0.231360,38.62,6.027700e+09,9.538330e+11,29.219866,1.72,121.01,1.0,1.0


In [285]:
df_candid.sort_values(by=['market_cap', 'relative_per', 'net_marketcap', 'fcf_marketcap'], ascending=[False, True, False, False])[['Symbol', 'sector', 'industry', 'price', 'market_cap', 'roe', 'relative_per', 'net_marketcap', 'fcf_marketcap', 'ev_ebitda']]

,Symbol,sector,industry,price,market_cap,roe,relative_per,net_marketcap,fcf_marketcap,ev_ebitda
0,NVDA,Technology,Semiconductors,175.02,4261212061696,0.266560,0.64,1.0,1.0,109.95
1,AAPL,Technology,Consumer Electronics,278.28,4129783545856,0.381321,1.28,1.0,1.0,117.92
2,MSFT,Technology,Software - Infrastructure,478.53,3556993138688,0.075600,1.81,1.0,1.0,74.67
3,TSM,Technology,Semiconductors,292.04,1514666852352,0.087308,0.44,30.0,9.0,-0.01
4,LLY,Healthcare,Drug Manufacturers - General,1027.51,921118310400,0.231360,1.72,1.0,1.0,121.01
5,ASML,Technology,Semiconductor Equipment & Materials,1080.85,419529424896,0.112812,0.79,1.0,0.0,152.06
6,PG,Consumer Defensive,Household & Personal Products,142.84,334314930176,0.089964,1.11,1.0,1.0,51.36
7,KLAC,Technology,Semiconductor Equipment & Materials,1193.92,157220798464,0.228900,0.77,1.0,1.0,109.21
8,GILD,Healthcare,Drug Manufacturers - General,120.40,149393162240,0.138411,0.64,2.0,3.0,36.44
9,SNY,Healthcare,Drug Manufacturers - General,48.68,118824140800,0.037170,0.54,2.0,3.0,31.37


---
#### **유행을 제거하고 가격이 재무로 설명할 수 있는 상태임. (대세 / 주류를 뽑는 섹터가 아님)**

- 같은 산업군으로 Peer 비교함 (per가 높은지 낮은지)

In [248]:
df_sector = (
    df.query("per > 0")
      .groupby(['sector', 'industry'])
      .per.mean()
      .reset_index(name='per_mean')
)

df_candid = df_candid.merge(df_sector, on=['sector', 'industry'], how='left')
df_candid['relative_per'] = np.round(df_candid['per'] / df_candid['per_mean'], 2)

- 2. ROE vs PBR로 정당한 가격 판단

In [249]:
# ROE (자기자본으로 얼마나 효율적으로 이익을 내는지 보는 지표)
# ROE = Net Income / Shareholders' Equity
# PBR = Market Cap / Shareholders' Equity
# Net Income = 당기순이익
# Shareholders' Equity = 자기자본(지분)

In [250]:
# “전체 시장에서, 내가 요구하는 수익률(r)을 기준으로 봤을 때 이 회사(혹은 이 집합)는 ‘충분한 이익을 벌고 있는가?’”

# Quality Premium Score = PBR / (ROE / (r - g))
# r = 요구수익률(10%)
# g = 장기성장률(3~5%)

# 음수, 무한대면 의미 없음
# roe는 '얼마나 잘 버는가', pbr은 '그 능력을 얼마나 사는가'
# per = pbr / roe

# qps가 0.5이하면 저평가, 0.5~1이면 정상, 1~1.2이면 살짝 비쌈, 1.2이상이면 비쌈

mask = (df['roe'] > 0) & (df['pbr'] > 0)
df_candid.loc[mask, 'relative_qps'] = df_candid.loc[mask, 'pbr'] * 0.07 / df_candid.loc[mask, 'roe']

- 3. FCF Yield 상대 점수 (Cash flow Score)
    - 잘 벌고 있느냐가 아니라 지금 당장 쓸수 있는 돈
    - FCF = Operating Cash Flow – Capital Expenditures
    - OCF : 영업에서 실제 들어온 돈, CAPEX : 공장, 서버, 장비에 쓴 돈

In [251]:
# 기업이 벌어들인 FCF이 시가총액 대비 몇 %인지 계산하는 것
# 1 이하 → 저평가
# 1~1.3 → 중립
# 1.3 이상 → 고평가

# fcf yield = free cash flow / market cap
mask = df_candid['fcf'] > 0

df_candid.loc[mask, 'relative_fcf_yield'] = (
    0.04 / (df_candid.loc[mask, 'fcf'] / df_candid.loc[mask, 'market_cap'])
).round(2)

- 4. EV/EBITDA 점수
    - 영업에서 벌 능력
    - 이 회사를 통째로 사서 영업으로 벌어들이는 현금을 몇 년만에 회수하는냐
    - EV = 시가총액 + 총부채 – 현금 및 현금성자산 (회사를 인수하는거니까 총부채도 가져와야 함)
    - EBITDA = 영업이익 + 감가삼각비 + 무형자산상각비
    - 결국 EV를 EBITDA로 나누고 이를 12로 나눠 월단위로 계산하는 것임

In [252]:
# 12 = 글로벌 적정 멀티플 중간값
# 1 이하 → 저평가
# 1~1.2 → 정상
# 1.2 이상 → 고평가

# EV = 시가총액 + 총부채 – 현금 및 현금성자산
# EBITDA = 영업이익 + 감가상각비 + 무형자산상각비

# 이 회사를 통째로 사서 영업으로 벌어들이는 현금으 몇 년만에 회수하는냐
mask = df_candid['ev_ebitda'] > 0
df_candid['value_multiple_score'] = df_candid.loc[mask, 'ev_ebitda'] / 12

- 5. 최종 결과

In [253]:
# FVS < 0.95 → 매우 저평가 (매수)
# 0.95 ≤ FVS ≤ 1.15 → 적정 가치 (보유)
# FVS > 1.15 → 고평가 (주의)
# FVS > 1.35 → 매우 고평가 (비중 축소)

df_candid['FVS'] = \
  0.30 * df_candid['relative_per'] + \
  0.30 * df_candid['relative_qps'] + \
  0.25 * df_candid['relative_fcf_yield'] + \
  0.15 * df_candid['value_multiple_score']


In [254]:
df_candid.sort_values(by=['relative_per', 'relative_qps'], ascending=True).head(1)

,Symbol,sector,industry,market_cap,net_income,per,net_margin,inventory_turnover,asset_turnover,leverage,total_equity,operating_cash_flow,capital_expenditure,debt,cash,ebitda,roe,pbr,fcf,ev,ev_ebitda,net_marketcap,fcf_marketcap,per_mean,relative_per,relative_qps,relative_fcf_yield,value_multiple_score,FVS
24,BYD,Consumer Cyclical,Resorts & Casinos,6823675904,1.439993e+09,3.787272,1.43,24.66,0.15,2.44,2.667313e+09,239982000.0,-145567000.0,2.567659e+09,319067000.0,1.949047e+09,0.52338,2.56,94415000.0,9.072268e+09,4.65,21.0,1.0,48.475628,0.08,NaN,2.89,0.3875,NaN
